# 08 — Generatore di telemetria sintetica CNC (pipeline ibrida SOTA)

Questo notebook **orchestra** il generatore di dati sintetici per il mandrino CNC.
La logica pesante vive nel pacchetto `synthgen/`; qui si segue il processo passo-passo:

1. Setup & configurazione
2. Caricamento dati reali + EDA rapida
3. Costruzione dataset (griglia, finestre, regimi, timing)
4. Modelli base: regime (Markov) + timing
5. Diffusione condizionata — **smoke test locale** (1-2 epoche, CPU)
6. Valutazione fedeltà locale (la "giuria")
7. **Submit** training completo su Azure ML (ritorna subito il `job_name`)
8. **Poll/stream** dello stato del job (ri-eseguibile)
9. **Download** del modello addestrato
10. Test locale: genera sintetico col modello cloud + scorecard vs reale

**Metodologia identica in locale e in cloud**: cambia solo la configurazione
(`local` vs `cloud`), non il codice. Le celle 7-9 sono idempotenti e indipendenti.

## 1 · Setup & configurazione

Carichiamo la config `local` da `configs/synthgen.yaml`. Per i loop veloci usiamo
un sottoinsieme di giorni e poche epoche.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Assicura che la repo root sia importabile (notebook in notebooks/).
REPO = Path.cwd()
while not (REPO / "synthgen").exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from synthgen.config import load_config, to_dict

cfg = load_config("local")
print("mode      :", cfg.mode)
print("device    :", cfg.device)
print("subset_days:", cfg.subset_days)
print("epochs    :", cfg.diffusion.epochs, "| timesteps:", cfg.diffusion.timesteps)
print("signals   :", cfg.data.signals)
print("out_path  :", cfg.out_path)

## 2 · Caricamento dati reali + EDA rapida

Una sola macchina, 4 segnali (`mandrino_load`, `mandrino_power`, `mandrino_torque`
+ il regime discreto `fase`). Campionamento irregolare sub-secondo.

In [ ]:
from synthgen.data import load_wide

raw = load_wide(cfg)
print("shape:", raw.shape)
print("periodo:", raw.index.min(), "->", raw.index.max())
display(raw.describe().T)
print("NaN per colonna:\n", raw.isna().sum())

In [ ]:
# Anteprima di un'ora di dati reali per capire dinamica e regimi.
sample = raw.loc[raw.index <= raw.index.min() + pd.Timedelta(hours=1)]
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
for ax, col in zip(axes, [*cfg.data.signals, cfg.data.regime_col]):
    ax.plot(sample.index, sample[col], lw=0.6)
    ax.set_ylabel(col)
axes[-1].set_xlabel("ts")
fig.suptitle("Telemetria reale — prima ora")
plt.tight_layout(); plt.show()

## 3 · Costruzione dataset

`build_dataset` produce le tre viste usate dai tre componenti:
griglia regolare + finestre (diffusione), sequenza `fase` (regime), gap inter-arrivo (timing).

In [ ]:
from synthgen.data import build_dataset, time_split_windows

ds = build_dataset(cfg)
print("finestre   :", ds.n_windows, "shape:", ds.windows.shape)
print("righe griglia:", len(ds.grid))
print("gap timing :", len(ds.gaps_s))
w_tr, r_tr, w_va, r_va, w_te, r_te = time_split_windows(ds, cfg)
print("split train/val/test:", len(w_tr), len(w_va), len(w_te))

## 4 · Modelli base: regime (Markov) + timing

Si fittano direttamente; sono veloci e robusti. Verifichiamo che il regime
campionato riproduca durate e transizioni del `fase` reale.

In [ ]:
from synthgen.models import RegimeMarkov, TimingModel
from synthgen.metrics import regime_durations, transition_matrix

regime = RegimeMarkov(
    n_states=cfg.diffusion.n_regimes,
    smoothing=cfg.regime.smoothing,
    min_dwell=cfg.regime.min_dwell,
).fit(ds.grid[cfg.data.regime_col].to_numpy())

timing = TimingModel(n_bins=cfg.timing.n_bins, max_gap_s=cfg.timing.max_gap_s).fit(
    ds.gaps_s, ds.gaps_regime
)

real_reg = ds.grid[cfg.data.regime_col].to_numpy()
synth_reg = regime.sample(len(real_reg), seed=0)
rd_real = regime_durations(real_reg)
rd_synth = regime_durations(synth_reg)
print("durata media regime reale/sintetico:", rd_real.mean().round(1), rd_synth.mean().round(1))

## 5 · Diffusione condizionata — smoke test locale

Addestriamo la diffusione per **poche epoche su CPU** solo per validare la
metodologia (loss che scende, sampling funzionante). Il training serio avviene
su Azure ML (celle 7-9).

In [ ]:
from synthgen.pipeline import fit, generate

# fit() addestra scaler + regime + timing + diffusione e salva gli artifact.
bundle = fit(cfg, ds=ds, save=True)
print("artifact salvati in:", cfg.out_path)

In [ ]:
# Generiamo una traccia sintetica e la confrontiamo visivamente col reale.
synth_wide = generate(bundle, n_steps=2000, seed=1, long_format=False)
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
for ax, col in zip(axes, cfg.data.signals):
    ax.plot(synth_wide.index, synth_wide[col], lw=0.6)
    ax.set_ylabel(col)
fig.suptitle("Telemetria SINTETICA (smoke test locale)")
plt.tight_layout(); plt.show()

## 6 · Valutazione fedeltà locale (la "giuria")

La scorecard quantifica quanto il sintetico assomiglia al reale: marginali (KS),
errore di correlazione, score discriminativo/predittivo, durate dei regimi, gap.
Con poche epoche i numeri saranno mediocri: è atteso, serve come baseline.

In [ ]:
from synthgen.metrics import fidelity_report

n = min(256, len(w_te))
synth_norm = bundle.diffusion.sample(r_te[:n], seed=1)
synth_real = np.stack([bundle.scaler.inverse_transform(s) for s in synth_norm])

rep = fidelity_report(
    w_te[:n].reshape(-1, len(cfg.data.signals)),
    synth_real.reshape(-1, len(cfg.data.signals)),
    names=list(cfg.data.signals),
    real_windows=w_te[:n],
    synth_windows=synth_real,
    real_regime=real_reg,
    synth_regime=synth_reg,
    n_states=cfg.diffusion.n_regimes,
)
pd.Series(rep.summary())

## 7 · Submit del training completo su Azure ML

Prepariamo uno snapshot self-contained (pacchetto `synthgen`, `configs/`, dati)
e lanciamo il job sul cluster GPU T4. **La cella ritorna subito** stampando il
`job_name`; il training prosegue in cloud.

> Richiede `az login` con accesso al workspace `anomalyml-mlw`.

In [ ]:
from synthgen import aml

code_dir = REPO / "cloud-training-synth" / "src"
conda_file = REPO / "cloud-training-synth" / "environment" / "conda.yml"

# Copia synthgen + configs + dati nello snapshot del job (usa la cfg locale,
# che conosce il path reale del parquet in _data_local).
aml.stage_cloud_bundle(cfg, code_dir)

# load_config('cloud') -> coordinate AML + iperparametri di training pieni.
cloud_cfg = load_config("cloud")
job_name = aml.submit_training(
    cloud_cfg,
    code_dir=code_dir,
    conda_file=conda_file,
)
print("JOB NAME:", job_name)

## 8 · Poll / stream dello stato del job

Cella **ri-eseguibile**: controlla lo stato senza bloccare. Imposta `job_name`
manualmente se riapri il notebook in una nuova sessione.

In [ ]:
# job_name = "<incolla-qui-se-nuova-sessione>"
status = aml.poll(cloud_cfg, job_name)
status

In [ ]:
# Facoltativo: stream bloccante dei log fino al termine.
# aml.stream(cloud_cfg, job_name)

## 9 · Download del modello addestrato

Scarica gli artifact da `outputs/` del run (scaler, regime, timing, diffusione,
`fidelity.json`) in locale via autenticazione AAD sul blob.

In [ ]:
model_dir = aml.download_model(cloud_cfg, job_name)
print("modello scaricato in:", model_dir)
for p in sorted(Path(model_dir).rglob("*")):
    print(" -", p.relative_to(model_dir))

## 10 · Test locale del modello cloud + scorecard finale

Ricarichiamo il bundle addestrato in cloud, generiamo telemetria sintetica e
produciamo la scorecard di fedeltà definitiva contro i dati reali di test.

In [ ]:
from synthgen.pipeline import SynthBundle

cloud_bundle = SynthBundle.load(cfg, Path(model_dir), device="cpu")

n = min(256, len(w_te))
synth_norm = cloud_bundle.diffusion.sample(r_te[:n], seed=2)
synth_real = np.stack([cloud_bundle.scaler.inverse_transform(s) for s in synth_norm])
synth_reg2 = cloud_bundle.regime.sample(len(real_reg), seed=2)

rep_final = fidelity_report(
    w_te[:n].reshape(-1, len(cfg.data.signals)),
    synth_real.reshape(-1, len(cfg.data.signals)),
    names=list(cfg.data.signals),
    real_windows=w_te[:n],
    synth_windows=synth_real,
    real_regime=real_reg,
    synth_regime=synth_reg2,
    n_states=cfg.diffusion.n_regimes,
)
pd.Series(rep_final.summary())

In [ ]:
# Confronto marginali reale vs sintetico (modello cloud).
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, c in zip(axes, range(len(cfg.data.signals))):
    ax.hist(w_te[:n].reshape(-1, len(cfg.data.signals))[:, c], bins=60, alpha=0.5, density=True, label="reale")
    ax.hist(synth_real.reshape(-1, len(cfg.data.signals))[:, c], bins=60, alpha=0.5, density=True, label="sintetico")
    ax.set_title(cfg.data.signals[c]); ax.legend()
plt.tight_layout(); plt.show()

### Esportazione in formato reale (long)

`generate(..., long_format=True)` produce lo schema dell'export reale
(`ts, signal_name, value, measure_id`), pronto per l'ingest in Fabric/KQL.

In [ ]:
long = generate(cloud_bundle, n_steps=5000, seed=3, long_format=True)
print(long.shape)
display(long.head(8))
out_csv = cfg.out_path / "synthetic_telemetry.parquet"
long.to_parquet(out_csv)
print("salvato:", out_csv)